# Final submission package, step 2 of 2: submission CSV (CPU)

Run on a **CPU runtime** after `reproduce_maps.ipynb`. Attach its output and the competition data. If both stages run in one session (see `pipeline.ipynb`), the maps in `/kaggle/working/maps` are used.

**Weights come from this repository.** `weights/` holds every file the pipeline loads; the three Mask2Former files are stored as numbered parts under 48 MB. On first use the parts are reassembled and every file is checked against the sha256 in `weights/repo_weights.json` and `weights/reproduction_weights.json`; any mismatch stops the run. No dataset needs to be attached for the weights. (Fallback only: without a `weights/` folder, the loader uses a mounted copy of the weights.)

**Test images.** `filseg.reproduction.paths.find_test_images()` tries the known Kaggle mount layouts of the competition data in order (`/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images` first), then accepts a single other `test_images` directory under `/kaggle/input`; otherwise it stops and lists what it found. The run then checks the 180 file names and every image sha256 against the pinned list.

From the maps to the CSV: the three Mask2Former maps are averaged and blended with the two U-Net maps (0.25 / 0.375 / 0.375); the blend is cut into filament candidates (SHIP_R3); a selector keeps or drops each candidate; each kept mask is repaired (re-thresholded inside its own window); the masks are run-length encoded into the competition CSV. Everything here is deterministic and runs on CPU.

**`MODE` chooses the CSV.** Left unset, the notebooks build the package default, `filseg.reproduction.modes.DEFAULT_MODE`; set `MODE` to any row of the table to build another. Each CSV is checked against its sha256. Every mode reads the same five probability maps; `"mftta"` (MF-TTA) also reads the Mask2Former D4 test-time-augmentation maps written by `reproduce_mf_views.ipynb` (Kaggle T4) and uses them in place of the three-fold Mask2Former mean everywhere downstream.

| `MODE` | Maps, selector and repair | CSV SHA-256 | Submission |
| --- | --- | --- | --- |
| `"pcombo"` | P-combo selector (68 features, cutoff 0.35) + P-cand repair | `35abbeed91108e40ddbd18c094c901d116441e363b10c31aa8092988d8e8f6e6` | 58278257 |
| `"mfrich"` | MF-rich selector (62 features, cutoff 0.375) + P-cand repair | `5711532ebb1360bdbc27a2bb04fd1f388b4a6494d0ab911067775aa10c018b85` | 58274749 |
| `"pneigh"` | P-neigh selector (60 features, cutoff 0.375) + P-cand repair | `c3edf2c8b681d42649dc7ccb6ed10e0400031b37c057d7d2798d6196bed0612c` | 58268408 |
| `"fresh_ctrl"` | Refit selector (54 features, cutoff 0.375) + P-cand repair | `b01866b3f849d7bc43534c188b91545c26f8cd839493175d00ff6d25fb19d085` | 58265972 |
| `"pcand"` | LR39 selector (39 features, cutoff 0.35) + P-cand repair | `14a5c1b684813081ad129d05e72fbb0c09592f292aeb1b1d84e121100c668c6b` | 58232010 |
| `"pnat"` | LR39 selector + P-nat repair (one threshold for every mask) | `d54257e7f8a1f815d442037050233e52859b0f9594f128b7b258f368f4d05fbe` | 58224552 |
| `"equivalent"` | LR39 selector, no repair (incumbent-equivalent) | `0b2123de5b614561ddf22c2bbc0f05000164ac6a595a9806104c7358b9f46b70` | 58224327 |
| `"mftta"` | MF-TTA map (8-view D4 mean of each MF fold model) + P-combo selector + P-cand repair | `3cdd3f06feec44b7b091d1807626c85dfbdcaf44afb285392e440b80432549ac` | 58299400 |
| `"ms125"` | Mask2Former crop-1536, 8-view + 1.25x scale (2 views); refit selector and repair; 5-fold U-Nets (final) | `f05ea0d7b6fc8f9345586b193c2866cd654f7fad1b838d609e41af5f46de80ed` | 58367893 |
| `"ms125d4"` | Mask2Former crop-1536, 8-view + 1.25x scale 8-view; refit selector and repair; 5-fold U-Nets | `c73318cca158dca7f0b9688de9a4fbc49ef0981e76c0acae140bdf8a589f950c` | 58368642 |
| `"ms4"` | Mask2Former crop-1536, 8-view + 1.25x 8-view + 1.5x 4-view; refit selector and repair; 5-fold U-Nets | `a3abcb959da41f8db2d895a06e290b8da821147b1b4c26af105cea12cda5ce24` | 58369061 |
| `"f1c15u"` | Mask2Former crop-1536, 8-view; refit selector and repair; 5-fold U-Nets | `af512e390bb4a365ff5ee0434c0ea2607e529759c0eab79d1c85b595027d395b` | 58360056 |
| `"c7c10d4_un5"` | Mask2Former crop-768 + crop-1024, 8-view; refit selector; 5-fold U-Nets | `573532cd29e5205a4c4e52792fcd4a801da41609d88c4e0a65d24c2f81cb9ad1` | 58338409 |
| `"c7c10_refit_un5"` | Mask2Former crop-768 + crop-1024; refit selector and repair; 5-fold U-Nets | `3134fef943bc0d7fd4f9ba7f016837cdf290a10b4f3b98777fa963079ec1bcfb` | 58337103 |
| `"c7c10_un5"` | Mask2Former crop-768 + crop-1024; refit selector; 5-fold U-Nets | `6efaef552809d5a89223a767ccba8b5cb97bab981562c1d0f28206b99af1f31a` | 58330231 |
| `"c7c10"` | Mask2Former crop-768 + crop-1024; refit selector; fold-2 U-Nets | `b11258ecd18d603c67b8db2870bae0d0cf47e24ff246964c0a80f3b0a43a59fe` | 58328986 |
| `"c768"` | Mask2Former crop-768; refit selector; fold-2 U-Nets | `167f756bdf5eecf4de56e5e208d9110b99ccf6e785b3a6f735ed1d5425bcddf3` | 58327493 |
| `"c768_w25"` | Mask2Former crop-768, blend weight 0.25; refit selector; fold-2 U-Nets | `2dd8156a4aeb06a34d63abda87ecccca87e7904f7cff5040dd98eb8f6567e463` | 58320909 |

**MF-map mode.** For `"mftta"` attach the output of `reproduce_mf_views.ipynb` as well (or run it in the same session, as `pipeline.ipynb` does). Its map set (full, local, MF0_tta, MF1_tta, MF2_tta) must have the sha256 `44ee26c7d4284b02103f005935611b0911d8f1fe20e1693bf73f0eb21fff9cc2`; the run stops otherwise.

**Chain modes** (`"ms125"` and the rows after `"mftta"`; `"ms125"` is our final submission). They read stored probability maps instead of the five maps above: the Mask2Former test maps of several fold models, input scales and views, and the U-Net test maps, from this repository for `"ms125"` (`weights/final/`) and from the private release files for the others (attach the release dataset; its files and hashes are listed in `src/filseg/reproduction/data/chain/release_files.json`). The Mask2Former map is the mean of the recipe's maps and is blended with weight w (0.55 for most), and each mode has its own instance filter and repair regressor (`src/filseg/reproduction/data/chain/<mode>.json`). Every stored map is hash-checked before use.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OPENBLAS_NUM_THREADS"] = "1"
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Clone this repository at its release tag (needs Internet), unless this notebook already runs inside a checkout.
SOURCE_TAG = "v1.1"
if not (REPO / "src/filseg/reproduction").is_dir():
    REPO = Path("/kaggle/working/filament-reproduction-source")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/Matapanino/filament-segmentation-2026-solution.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_TAG], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-reproduce-cpu.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))  # the editable install is seen only by new processes; later cells import filseg here

In [ ]:
from filseg.reproduction.modes import DEFAULT_MODE, resolve
from filseg.reproduction.paths import find_release, find_test_images, find_variant_maps
MODE = globals().get("MODE") or DEFAULT_MODE  # the CSV to build (table above); pipeline.ipynb sets MODE before running this notebook
MODE, MODE_ROW = resolve(MODE)  # stops on a mode without a pinned CSV
IMAGE_ROOT = find_test_images()  # or set IMAGE_ROOT = Path("<the test JPEG directory>")
OUTPUT = Path("/kaggle/working/submission_" + MODE)
command = [sys.executable, "-m", "filseg.reproduction", "csv", "--images", str(IMAGE_ROOT), "--output", str(OUTPUT), "--mode", MODE]
VARIANT_MAPS = None
if "recipe" in MODE_ROW["kwargs"]:
    # A chain mode reads stored maps (no GPU stage): the final submission's are in this repository (weights/final),
    # the other chain modes' in the release dataset (attach it).
    from filseg.reproduction.chain import repo_release
    RELEASE = repo_release(MODE) or find_release()
    assert RELEASE is not None, "MODE " + MODE + " reads stored maps: attach the release dataset (the folder with release_manifest.json)"
    MAPS_ROOT = RELEASE
    command += ["--release", str(RELEASE)]
else:
    # Maps from this session (pipeline.ipynb) first, otherwise one attached reproduce_maps output.
    map_results = sorted(Path("/kaggle/working/maps").glob("inference_result.json")) or sorted(Path("/kaggle/input").rglob("inference_result.json"))
    assert len(map_results) == 1, "Run reproduce_maps.ipynb first, or attach exactly one completed reproduce_maps output"
    MAPS_ROOT = map_results[0].parent
    command += ["--maps", str(MAPS_ROOT)]
    # An MF-map mode also reads the Mask2Former D4 TTA maps of reproduce_mf_views.ipynb: this session's first, otherwise one attached output.
    VARIANT = MODE_ROW["kwargs"].get("mf_variant")
    if VARIANT:
        VARIANT_MAPS = find_variant_maps(VARIANT)
        assert VARIANT_MAPS is not None, "MODE " + MODE + " needs the MF-view maps: run reproduce_mf_views.ipynb on a T4 (pipeline.ipynb does), or attach its output"
        command += ["--variant-maps", str(VARIANT_MAPS)]
subprocess.run(command, check=True)
result = json.loads((OUTPUT / "csv_result.json").read_text())
print({k: result[k] for k in ("status", "images", "candidates", "kept", "csv")})

In [ ]:
from filseg.reproduction.validate import validate_submission
from filseg.reproduction.pipeline import test_keys
check = validate_submission(OUTPUT / "submission.csv", test_keys())
print({k: check[k] for k in ("passed", "rows", "files_with_rows", "empty_rows", "overlapping_pixel_images", "sha256")})
print({"mode": MODE, "expected_sha256": MODE_ROW["sha256"], "matches_the_expected_csv": check["sha256"] == MODE_ROW["sha256"]})